In [4]:
!pip install window_ops
!pip install statsforecast
!pip install mlforecast
!pip install lightgbm

In [5]:
import pandas as pd
import numpy as np

from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

from mlforecast.target_transforms import Differences
from window_ops.rolling import rolling_mean

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape, bias

In [6]:
df = pd.read_parquet('/content/sample_hotels-1.parquet')

df['ds'] = pd.to_datetime(df['ds'])

otb_cols = [f'otb_{i}' for i in range(28, 61) if f'otb_{i}' in df.columns]

df = df[['unique_id', 'ds', 'y', 'location_type', 'hotel_type'] + otb_cols]

df = df.query("unique_id not in ['hotel_77', 'hotel_28']")

In [7]:
#  the training set first
train_df = df.groupby('unique_id').head(-28).reset_index(drop=True)

#  dummy variables for categorical predictors
train_df_dummies = df.groupby('unique_id').apply(lambda x: x.iloc[:-28]).reset_index(drop=True)

# Identify the names of these new dummy columns for later use
dummies = [c for c in train_df_dummies if 'location_type_' in c or 'hotel_type_' in c]

/tmp/ipykernel_7178/3403269105.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df_dummies = df.groupby('unique_id').apply(lambda x: x.iloc[:-28]).reset_index(drop=True)


In [8]:
ml_models = {
    'LGBM': LGBMRegressor(random_state=42, verbosity=-1),
    'RF': RandomForestRegressor(random_state=42)
}

In [9]:
ml = MLForecast(
    models=ml_models,
    freq='D',
    lags=[28, 29, 35],
    lag_transforms={
        28: [(rolling_mean, 7)]
    },
    target_transforms=[Differences([7])]


)

In [10]:
key_predictors = ['otb_28'] + dummies

cv_ml = ml.cross_validation(
    df=train_df_dummies[['unique_id', 'ds', 'y'] + key_predictors],
    h=28,
    n_windows=5,
    step_size=28,
    static_features=[]  # dummies are static, otb_28 is dynamic
)

In [11]:
eval_ml = evaluate(
    df = cv_ml,
    metrics = [mae, rmse, bias],
    models=['RF', 'LGBM']
)
eval_ml = eval_ml.drop(columns=['cutoff']).groupby(['unique_id','metric']).mean().reset_index()
eval_ml

,unique_id,metric,RF,LGBM
0,hotel_0,bias,0.061236,0.052970
1,hotel_0,mae,0.207827,0.206011
2,hotel_0,rmse,0.247166,0.247536
3,hotel_105,bias,-0.078062,-0.083798
4,hotel_105,mae,0.144859,0.150690
5,hotel_105,rmse,0.180292,0.188291
6,hotel_112,bias,-0.063483,-0.056177
7,hotel_112,mae,0.131028,0.132579
8,hotel_112,rmse,0.157224,0.159616
9,hotel_126,bias,0.045586,0.052239


In [12]:
#without mape
eval_ml = evaluate(
    df = cv_ml,
    metrics = [mae, rmse, bias],
    models=['RF', 'LGBM']
)
eval_ml = eval_ml.drop(columns=['cutoff']).groupby(['unique_id','metric']).mean().reset_index()
eval_ml

,unique_id,metric,RF,LGBM
0,hotel_0,bias,0.061236,0.052970
1,hotel_0,mae,0.207827,0.206011
2,hotel_0,rmse,0.247166,0.247536
3,hotel_105,bias,-0.078062,-0.083798
4,hotel_105,mae,0.144859,0.150690
5,hotel_105,rmse,0.180292,0.188291
6,hotel_112,bias,-0.063483,-0.056177
7,hotel_112,mae,0.131028,0.132579
8,hotel_112,rmse,0.157224,0.159616
9,hotel_126,bias,0.045586,0.052239


In [13]:
#with mape
eval_ml = evaluate(
    df = cv_ml,
    metrics = [mae, rmse,mape, bias],
    models=['RF', 'LGBM']
)
eval_ml = eval_ml.drop(columns=['cutoff']).groupby(['unique_id','metric']).mean().reset_index()
eval_ml

,unique_id,metric,RF,LGBM
0,hotel_0,bias,0.061236,0.052970
1,hotel_0,mae,0.207827,0.206011
2,hotel_0,mape,0.328947,0.327884
3,hotel_0,rmse,0.247166,0.247536
4,hotel_105,bias,-0.078062,-0.083798
...,...,...,...,...
63,hotel_91,rmse,0.237551,0.235396
64,hotel_98,bias,-0.036707,-0.036555
65,hotel_98,mae,0.158589,0.156084
66,hotel_98,mape,0.779299,0.752020


In [14]:
eval_ml['best_model'] = eval_ml[['RF', 'LGBM']].idxmin(axis=1)

In [15]:
win_counts = (
    eval_ml
    .groupby(['metric', 'best_model'])
    .size()
    .reset_index(name='wins')
)

display(win_counts)

,metric,best_model,wins
0,bias,LGBM,7
1,bias,RF,10
2,mae,LGBM,12
3,mae,RF,5
4,mape,LGBM,11
5,mape,RF,6
6,rmse,LGBM,11
7,rmse,RF,6


In [16]:
ml_models = ['RF', 'LGBM']

In [17]:
metrics = ['mae', 'rmse', 'mape', 'bias']
summary_list = []

for m in metrics:
    wins = (
        eval_ml.query(f"metric == '{m}'")
        .assign(winner = lambda x: x[ml_models].idxmin(axis=1))
        ['winner'].value_counts()
    )
    summary_list.append(wins.rename(m.upper()))

# Merge into one summary DataFrame
win_summary_table = pd.concat(summary_list, axis=1).fillna(0).astype(int)
display(win_summary_table)

,MAE,RMSE,MAPE,BIAS
winner,,,,
LGBM,12,11,11,7
RF,5,6,6,10


##Tabpfn

In [23]:
#!pip install tabpfn

In [27]:
from tabpfn import TabPFNRegressor
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences

# 1. Initialize with specific arguments to avoid the "missing arguments" error
# TabPFN often expects a 'device' and 'n_estimators' or similar depending on version
# Use 'cuda' to specify GPU usage
tpfn_model = TabPFNRegressor(device='cuda')

# 2. Put it in your MLForecast object
fcst = MLForecast(
    models={
        'lgbm': LGBMRegressor(verbosity=-1),
        'tabpfn': tpfn_model  # Give it a string name in a dictionary
    },
    freq='D',
    lags=[1, 7, 14],
    date_features=['dayofweek', 'month']
)

In [20]:
#!pip install tabpfn-client

In [28]:
# Convert object columns to categories so LightGBM can process them
train_df['location_type'] = train_df['location_type'].astype('category')
train_df['hotel_type'] = train_df['hotel_type'].astype('category')

# Set the TabPFN license token
import os
# Replace 'YOUR_API_KEY_HERE' with your actual API key obtained from https://ux.priorlabs.ai/account
os.environ["TABPFN_TOKEN"] = "YOUR_API_KEY_HERE" # not adding to public github, but created our own
# Allow TabPFN to run on CPU with large datasets
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

# Now run your CV
cv_results = fcst.cross_validation(
    df=train_df,
    h=28,
    step_size=28,
    n_windows=5,
    static_features=['location_type', 'hotel_type'] # Include them here now that they are categories
)

In [31]:
evaluation_df = evaluate(
    cv_results,
    metrics=[mae, rmse],
    id_col='unique_id',
    target_col='y'
)

In [32]:
display(evaluation_df)

,unique_id,cutoff,metric,lgbm,tabpfn
0,hotel_0,2023-01-13,mae,0.114761,0.118769
1,hotel_105,2023-01-13,mae,0.099072,0.096518
2,hotel_112,2023-01-13,mae,0.068157,0.066432
3,hotel_126,2023-01-13,mae,0.123044,0.114021
4,hotel_133,2023-01-13,mae,0.109622,0.092735
...,...,...,...,...,...
165,hotel_7,2023-05-05,rmse,0.133192,0.125518
166,hotel_70,2023-05-05,rmse,0.174646,0.155329
167,hotel_84,2023-05-05,rmse,0.106588,0.104052
168,hotel_91,2023-05-05,rmse,0.117101,0.125570


In [33]:
# save to csv
evaluation_df.to_csv('TabPFN_output.csv', index=False)